# Phase 2 — N1 kiểm tra gói baseline tái lập

Notebook được thực thi thật trên Python 3.14.3 bằng compile/exec, cùng namespace và lưu stdout.
Chỉ đọc báo cáo đã khóa và kiểm inference validation; không train, không đánh giá lại test.


In [1]:
from pathlib import Path
import json
import platform
import sys
import pandas as pd

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "src/model").is_dir() else HERE.parent
assert (ROOT / "src/model/reporting.py").is_file(), "Run from the repo root or notebooks/"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.model.reporting import read_metrics, summary_table, component_table, per_class_table
from src.model.baseline import load_artifact
from src.model.verify_handoff import verify_handoff

import os
RUN = Path(os.environ.get("T08_MODEL_RUN", str(ROOT / "data/processed/p2_n1_reproduction_20261017"))).resolve()
metrics = read_metrics(RUN / "reports/model_metrics.json")
manifest = json.loads((RUN / "docs/handoffs/phase-2-model.json").read_text(encoding="utf-8"))
assert metrics["status"] == "complete"
assert manifest["status"] == "final_evaluated"
assert manifest["lock"]["test_status"] == "complete"
assert metrics["dataset_version"] == manifest["dataset_version"]
print("Python:", platform.python_version())
print("Dataset:", metrics["dataset_version"])
print("Final evaluation:", metrics["status"])
print("Models:", manifest["model_versions"])


Python: 3.14.3
Dataset: t08-v1.0-20261010
Final evaluation: complete
Models: {'dummy': 'tfidf-v1-n1-reproduction-20261017-dummy', 'tfidf': 'tfidf-v1-n1-reproduction-20261017'}


## Kiểm tra toàn vẹn và tái chạy cục bộ

Kiểm SHA-256/byte của các file bàn giao **trước khi** nạp artifact tin cậy của nhóm.
`verify_handoff` chỉ suy luận lại 5 ví dụ validation đã đóng gói với usage=demo;
không tính lại test metric. Báo cáo này xác minh máy hiện tại; tái chạy độc lập
trên máy thứ hai của N1 và review lỗi của N2 vẫn giữ trạng thái riêng.


In [2]:
verification = verify_handoff(repo_root=RUN)
saved_verification = json.loads((RUN / "reports/reproduction_verification.json").read_text(encoding="utf-8"))
assert verification["result"] == "PHASE2 HANDOFF VERIFIED"
assert verification["artifact_sha256"] == metrics["artifact_sha256"] == saved_verification["artifact_sha256"]
assert verification["reproduction_smoke"]["all_predictions_match"]
assert verification["runtime_versions_match"]
print(json.dumps(verification, ensure_ascii=False, indent=2))
print("N1 second machine:", manifest["n1_second_machine_reproduction"])
print("N2 validation review:", manifest["n2_validation_review"])

artifact_path = RUN / manifest["lock"]["artifact"].replace("\\", "/")
artifact = load_artifact(artifact_path)
assert artifact["dataset_version"] == metrics["dataset_version"]
assert artifact["model_versions"] == manifest["model_versions"]
print("Artifact loaded:", artifact_path.name)
print("Fallback heads:", artifact["fallback_metrics"])


{
  "result": "PHASE2 HANDOFF VERIFIED",
  "verification_scope": "local_offline_reproduction",
  "dataset_version": "t08-v1.0-20261010",
  "model_versions": {
    "dummy": "tfidf-v1-n1-reproduction-20261017-dummy",
    "tfidf": "tfidf-v1-n1-reproduction-20261017"
  },
  "artifact_sha256": "c68d530ff7818f5d469da647246e787517d9d213b59286d9a6f67dd647d05265",
  "files_verified": 40,
  "library_versions": {
    "python": "3.14.3",
    "scikit-learn": "1.9.1",
    "joblib": "1.6.0",
    "numpy": "2.5.3",
    "pandas": "3.0.6",
    "scipy": "1.18.1",
    "PyYAML": "6.0.3",
    "pyarrow": "25.0.1"
  },
  "runtime_library_versions": {
    "python": "3.14.3",
    "scikit-learn": "1.9.1",
    "joblib": "1.6.0",
    "numpy": "2.5.3",
    "pandas": "3.0.6",
    "scipy": "1.18.1",
    "PyYAML": "6.0.3",
    "pyarrow": "25.0.1"
  },
  "runtime_versions_match": true,
  "reproduction_smoke": {
    "cases_verified": 5,
    "model_predictions_verified": 10,
    "prediction_usage": "demo",
    "all_predic

## Chọn cấu hình bằng validation trước khi mở test

Hai ứng viên dùng chung split, cùng quy tắc che chuỗi CVSS rõ ràng trong mô tả.
Tiêu chí đã khai báo: trung bình macro-F1 của tám metric trên validation, giữ đủ
label map; nếu hòa trong ngưỡng thì ưu tiên chi phí huấn luyện thấp hơn. Bảng dưới
đọc lịch sử lựa chọn đã khóa; không chọn lại từ kết quả test.


In [3]:
selection = json.loads((ROOT / "reports/model_selection.json").read_text(encoding="utf-8"))
assert selection["status"] == "locked" and selection["chosen_candidate"] == 1
assert artifact["selection"]["selected_C"] == 1.0
assert artifact["selection"]["new_parameter_search"] is False
selection_rows = pd.DataFrame([
    {"candidate": r["candidate"], "C": r["overrides"]["C"],
     "validation_mean_macro_f1": r["validation_mean_macro_f1"]}
    for r in selection["candidates"]])
print("N3 historical selection, not a new search:")
print(selection_rows.to_string(index=False))
print("N1 fixed configuration:", artifact["selection"])
print("Locked:", manifest["lock"]["locked_at"])
print("Test started:", manifest["lock"]["test_started_at"])


N3 historical selection, not a new search:
 candidate   C  validation_mean_macro_f1
         1 1.0                  0.750431
         2 0.5                  0.744926
N1 fixed configuration: {'status': 'fixed_reproduction_of_N3_selected_candidate', 'selected_C': 1.0, 'new_parameter_search': False, 'original_source_commit': 'a5c0c16d862059819c2b10f973e91827796ac30e'}
Locked: 2026-10-16T17:25:02.093653+00:00
Test started: 2026-10-16T17:25:44.542352+00:00


## Validation — bảng tổng hợp

`all_ml` là toàn bộ cohort ML; `in_scope` là Web/mobile theo quyết định scope đã khóa.
Macro-F1 dùng mọi lớp khai báo, `zero_division=0`. Exact-vector đòi hỏi đúng cả tám
metric và prediction hợp lệ. MAE/RMSE chỉ tính trên cặp score hợp lệ; luôn giữ mẫu
số/coverage và số lỗi. Validation scoped có 69 CVE; không có mobile trong split này.


In [4]:
validation_summary = summary_table(metrics, "validation")
print(validation_summary.to_string(index=False, float_format=lambda value: f"{value:.6f}"))


model    scope     n  mean_macro_f1  exact_vector_accuracy  base_mae  base_rmse  score_pairs  score_denominator  invalid_predictions
dummy   all_ml 19606       0.298628               0.101755  2.936030   3.391720        19606              19606                    0
dummy in_scope    69       0.278970               0.043478  3.388406   3.727804           69                 69                    0
tfidf   all_ml 19606       0.750431               0.383709  1.118805   1.837327        19606              19606                    0
tfidf in_scope    69       0.616601               0.231884  1.639130   2.474420           69                 69                    0


## Test cuối — bảng tổng hợp riêng

Kết quả test được ghi sau khi khóa lựa chọn/artifact. Notebook chỉ đọc báo cáo đó.
Scoped test có 108 CVE Web; cả tám CVE mobile của cohort đều ở train, vì vậy không
công bố chất lượng riêng trên mobile. Cỡ mẫu scoped nhỏ hơn nhiều so với toàn miền.


In [5]:
test_summary = summary_table(metrics, "test")
print(test_summary.to_string(index=False, float_format=lambda value: f"{value:.6f}"))


model    scope     n  mean_macro_f1  exact_vector_accuracy  base_mae  base_rmse  score_pairs  score_denominator  invalid_predictions
dummy   all_ml 18821       0.298403               0.102279  2.900553   3.355398        18821              18821                    0
dummy in_scope   108       0.289889               0.083333  3.257407   3.645037          108                108                    0
tfidf   all_ml 18821       0.752142               0.386536  1.078030   1.769874        18821              18821                    0
tfidf in_scope   108       0.654889               0.351852  1.750926   2.833023          108                108                    0


## TF-IDF — từng metric trên validation


In [6]:
validation_components = component_table(metrics, "validation", model_kind="tfidf")
print(validation_components.to_string(index=False, float_format=lambda value: f"{value:.6f}"))


   scope metric     n  accuracy  macro_f1
  all_ml     AV 19606  0.890085  0.703958
  all_ml     AC 19606  0.895644  0.617856
  all_ml     PR 19606  0.741508  0.672823
  all_ml     UI 19606  0.931756  0.920737
  all_ml      S 19606  0.940426  0.911736
  all_ml      C 19606  0.791951  0.784841
  all_ml      I 19606  0.805162  0.806219
  all_ml      A 19606  0.795981  0.585275
in_scope     AV    69  1.000000  0.500000
in_scope     AC    69  0.913043  0.601923
in_scope     PR    69  0.681159  0.453268
in_scope     UI    69  0.797101  0.783989
in_scope      S    69  0.840580  0.759886
in_scope      C    69  0.652174  0.628980
in_scope      I    69  0.710145  0.683810
in_scope      A    69  0.652174  0.520952


## TF-IDF — từng metric trên test


In [7]:
test_components = component_table(metrics, "test", model_kind="tfidf")
print(test_components.to_string(index=False, float_format=lambda value: f"{value:.6f}"))


   scope metric     n  accuracy  macro_f1
  all_ml     AV 18821  0.899208  0.672052
  all_ml     AC 18821  0.908188  0.637872
  all_ml     PR 18821  0.746028  0.679390
  all_ml     UI 18821  0.921205  0.907135
  all_ml      S 18821  0.941555  0.916197
  all_ml      C 18821  0.808618  0.801482
  all_ml      I 18821  0.818926  0.820244
  all_ml      A 18821  0.794485  0.582761
in_scope     AV   108  0.972222  0.346445
in_scope     AC   108  0.898148  0.473171
in_scope     PR   108  0.722222  0.595146
in_scope     UI   108  0.925926  0.881773
in_scope      S   108  0.962963  0.933333
in_scope      C   108  0.675926  0.665550
in_scope      I   108  0.777778  0.740474
in_scope      A   108  0.796296  0.603218


## TF-IDF — F1/support từng lớp trên validation

Lớp support 0 vẫn xuất hiện; không gộp lớp hiếm hoặc bỏ khỏi macro-F1.


In [8]:
validation_classes = per_class_table(metrics, "validation", model_kind="tfidf")
assert len(validation_classes) == 2 * sum(len(labels) for labels in artifact["label_order"].values())
print(validation_classes.to_string(index=False, float_format=lambda value: f"{value:.6f}"))
print("Zero-support rows:", int(validation_classes.support.eq(0).sum()))


   scope metric label  precision   recall       f1  support
  all_ml     AV     N   0.920355 0.938984 0.929576    13685
  all_ml     AV     A   0.636516 0.547550 0.588691      694
  all_ml     AV     L   0.851138 0.814560 0.832448     5096
  all_ml     AV     P   0.411765 0.534351 0.465116      131
  all_ml     AC     L   0.961728 0.926278 0.943671    18502
  all_ml     AC     H   0.236282 0.382246 0.292042     1104
  all_ml     PR     N   0.763130 0.824738 0.792739     9637
  all_ml     PR     L   0.762638 0.691371 0.725258     8379
  all_ml     PR     H   0.499687 0.501258 0.500471     1590
  all_ml     UI     N   0.947123 0.953478 0.950290    13413
  all_ml     UI     R   0.897755 0.884709 0.891184     6193
  all_ml      S     U   0.946803 0.977813 0.962058    15144
  all_ml      S     C   0.915280 0.813537 0.861414     4462
  all_ml      C     N   0.704336 0.753954 0.728301     4869
  all_ml      C     L   0.869847 0.758517 0.810376     5313
  all_ml      C     H   0.801762 0.83043

## TF-IDF — F1/support từng lớp trên test


In [9]:
test_classes = per_class_table(metrics, "test", model_kind="tfidf")
assert len(test_classes) == 2 * sum(len(labels) for labels in artifact["label_order"].values())
print(test_classes.to_string(index=False, float_format=lambda value: f"{value:.6f}"))
print("Zero-support rows:", int(test_classes.support.eq(0).sum()))


   scope metric label  precision   recall       f1  support
  all_ml     AV     N   0.926169 0.949869 0.937869    13365
  all_ml     AV     A   0.435616 0.287523 0.346405      553
  all_ml     AV     L   0.874037 0.836812 0.855019     4743
  all_ml     AV     P   0.485577 0.631250 0.548913      160
  all_ml     AC     L   0.965384 0.936542 0.950744    17807
  all_ml     AC     H   0.269082 0.410256 0.325000     1014
  all_ml     PR     N   0.764223 0.840145 0.800387     8858
  all_ml     PR     L   0.791026 0.689522 0.736795     8284
  all_ml     PR     H   0.476369 0.528291 0.500988     1679
  all_ml     UI     N   0.939653 0.946940 0.943282    13023
  all_ml     UI     R   0.878708 0.863401 0.870987     5798
  all_ml      S     U   0.940440 0.985191 0.962295    14248
  all_ml      S     C   0.945828 0.805598 0.870099     4573
  all_ml      C     N   0.710366 0.775805 0.741645     4505
  all_ml      C     L   0.883958 0.787160 0.832755     5187
  all_ml      C     H   0.823206 0.83700

## Support train và audit che chuỗi CVSS

Support được đọc từ gói Phase 1. TF-IDF/weights chỉ fit train; bảng support không
biến CVSS/EPSS/KEV thành feature. Audit che chuỗi dùng cùng policy cố định ở mọi split;
không xóa từ nghiệp vụ như remote/authenticated. Các số ở đây đọc từ artifact đã khóa.


In [10]:
release_folder = ROOT / "data/processed/phase1_v1_20261010"
label_support = pd.read_csv(release_folder / "label_support.csv")
print(label_support.loc[label_support["split"].eq("train")].to_string(index=False))
print("Preprocessing policy:", artifact["preprocessing"]["policy_version"])
from src.model.baseline import preprocess_descriptions
descriptions = pd.read_parquet(release_folder / "ml_cohort.parquet", columns=["split", "description"])
audits = {split: preprocess_descriptions(descriptions.loc[descriptions.split.eq(split), "description"])[1]
          for split in ("train", "validation", "test")}
print(pd.DataFrame(audits).T.to_string())
print("Structural text audit only; no prediction or test error inspection.")


       cohort split metric label  support  size
       all_ml train     AV     N    21730 28815
       all_ml train     AV     A      491 28815
       all_ml train     AV     L     6378 28815
       all_ml train     AV     P      216 28815
       all_ml train     AC     L    27790 28815
       all_ml train     AC     H     1025 28815
       all_ml train     PR     N    16123 28815
       all_ml train     PR     L     9746 28815
       all_ml train     PR     H     2946 28815
       all_ml train     UI     N    18691 28815
       all_ml train     UI     R    10124 28815
       all_ml train      S     U    22904 28815
       all_ml train      S     C     5911 28815
       all_ml train      C     N     5491 28815
       all_ml train      C     L     6794 28815
       all_ml train      C     H    16530 28815
       all_ml train      I     N     8087 28815
       all_ml train      I     L     6885 28815
       all_ml train      I     H    13843 28815
       all_ml train      A     N    1252

## Mẫu lỗi validation

N2 đã phân tích 13 CVE trong báo cáo review. Manifest ZIP giữ trạng thái lúc đóng gói; nhật ký tích hợp cập nhật riêng.


In [11]:
queue = pd.read_csv(RUN / "reports/validation_errors_n2.csv",
                    dtype={"cve_id": "string", "split": "string", "dataset_version": "string", "model_version": "string"})
assert len(queue) == 10 and queue["split"].eq("validation").all()
assert queue.dataset_version.eq(metrics["dataset_version"]).all()
assert queue.model_version.eq(artifact["model_versions"]["tfidf"]).all()
queue_columns = ["cve_id", "split", "cvss_base_score", "predicted_base_score", "prediction_error"]
print(queue.loc[:, queue_columns].fillna("").to_string(index=False))
print("Review status:", manifest["n2_validation_review"])

print("N2 completed review: reports/p2_validation_security_review.md (13 CVEs)")


         cve_id      split  cvss_base_score  predicted_base_score prediction_error
 CVE-2010-10011 validation              7.5                   6.5                 
  CVE-2012-6664 validation              9.1                   7.5                 
CVE-2014-125110 validation              3.5                   6.1                 
CVE-2014-125111 validation              3.5                   6.1                 
 CVE-2015-10123 validation              8.8                   8.8                 
 CVE-2015-10129 validation              5.9                   2.2                 
 CVE-2015-10131 validation              3.5                   6.1                 
 CVE-2015-10132 validation              3.5                   6.1                 
 CVE-2016-15038 validation              6.5                   9.8                 
 CVE-2016-20021 validation              9.8                   9.1                 
Review status: pending_independent_review
N2 completed review: reports/p2_validation_se

## Giới hạn sử dụng

Mô hình ước lượng severity từ mô tả; kết quả không quyết định thứ tự vá và không thay
nhãn quan sát. Split theo thời gian trên snapshot NVD hiện có không khôi phục tri thức
lịch sử tại ngày công bố. Mô tả/nhãn có thể được cập nhật muộn, MinHash có thể bỏ sót
trùng gần, và 5.944 nhóm sản phẩm xuất hiện xuyên split. Không suy ra tổng quát hóa
sang sản phẩm chưa gặp hoặc hiệu năng riêng trên mobile.

DistilBERT được ghi theo trạng thái thực; baseline không chứng minh mô hình đó tốt
hơn hay kém hơn. Đối chiếu nghiên cứu và thông tin thiết bị xem `docs/model-card.md`.


In [12]:
print("DistilBERT:", json.dumps(metrics["distilbert"], ensure_ascii=False))
print("Local reproduction:", saved_verification["result"])
print("N1 independent second machine:", manifest["n1_second_machine_reproduction"])
print("Notebook finished: read-only artifact analysis; no fit or final-test re-evaluation.")


DistilBERT: {"status": "not_run", "reason": "D05 pending"}
Local reproduction: PHASE2 HANDOFF VERIFIED
N1 independent second machine: retrained_locally_original_artifact_unavailable
Notebook finished: read-only artifact analysis; no fit or final-test re-evaluation.


## DistilBERT — thử nghiệm bổ sung một epoch cố định
Đọc báo cáo đã khóa, không chọn lại cấu hình từ điểm test.


In [13]:
from src.model.predict_distilbert import verify_bundle
DISTIL = ROOT / "data/processed/p2_distilbert_20261017"
distil_manifest = verify_bundle(DISTIL)
distil_metrics = read_metrics(DISTIL / "metrics.json")
for split in ("validation", "test"):
    print(split)
    print(summary_table(distil_metrics, split).to_string(index=False))
    print(component_table(distil_metrics, split, model_kind="distilbert").to_string(index=False))
    print(per_class_table(distil_metrics, split, model_kind="distilbert").to_string(index=False))
print("Truncation:", distil_metrics["token_audit"])
print("Training:", distil_metrics["training"])


validation
     model    scope     n  mean_macro_f1  exact_vector_accuracy  base_mae  base_rmse  score_pairs  score_denominator  invalid_predictions
distilbert   all_ml 19606       0.734138               0.336887  1.185744   1.939135        19606              19606                    0
distilbert in_scope    69       0.527830               0.159420  1.659420   2.648024           69                 69                    0
   scope metric     n  accuracy  macro_f1
  all_ml     AV 19606  0.888248  0.722235
  all_ml     AC 19606  0.858564  0.570012
  all_ml     PR 19606  0.719831  0.646376
  all_ml     UI 19606  0.920331  0.907887
  all_ml      S 19606  0.938080  0.908731
  all_ml      C 19606  0.784199  0.777575
  all_ml      I 19606  0.770835  0.771510
  all_ml      A 19606  0.806692  0.568775
in_scope     AV    69  0.913043  0.300962
in_scope     AC    69  0.956522  0.488889
in_scope     PR    69  0.637681  0.433974
in_scope     UI    69  0.710145  0.691413
in_scope      S    69  0.7826